# Best vs Overfit: Attribution Denoising

Compare DeepLIFT/SHAP logos from the best-validated model vs an overfit checkpoint.
Row 3 = `best - (overfit - best)` — subtract the "noise" the overfit model learned.

In [1]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
_interp = os.path.abspath(os.path.join(os.getcwd(), '..'))
for _p in [_interp, os.path.join(_interp, 'tangermeme')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from alphagenome_pytorch import AlphaGenome
from alphagenome_pytorch.extensions.finetuning.transfer import remove_all_heads
from alphagenome_pytorch.extensions.finetuning.utils import sequence_to_onehot
from tangermeme.plot import plot_logo
from tangermeme.deep_lift_shap import deep_lift_shap, _nonlinear
from ag_deeplift_patches import patch_alphagenome, AGCustomGELU

patch_alphagenome()

AlphaGenome patches applied (all functional activations -> nn.Module).


In [ ]:
# --- Config ---
CELL_TYPE = 'K562'
MODEL_NAME = 'K562_overfit_study'
BEST_CKPT = 'best_stage2.pt'
OVERFIT_CKPT = 'latest_stage2.pt'  # same run, past early stopping
N_SEQS = 5
N_SHUFFLES = 20

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
RESULTS_DIR = os.path.join(REPO_ROOT, 'training', 'results')
WEIGHTS_PATH = os.path.join(REPO_ROOT, 'weights', 'model_fold_0.safetensors')
DATA_DIR = '/grid/wsbs/home_norepl/pmantill/LentiMPRA_mcs/LentiMoCon/test_run_lenti_data'

PROMOTER_SEQ = 'TCCATTATATACCCTCTAGTGTCGGTTCACGCAATG'
RAND_BARCODE = 'AGAGACTGAGGCCAC'
ENCODER_DIM = 1536

In [3]:
import pandas as pd
df = pd.read_csv(os.path.join(DATA_DIR, f'{CELL_TYPE}.tsv'), sep='\t')
df = df[(df['rev'] == 0) & (df['fold'] == 10)].nlargest(N_SEQS, 'mean_value')

constructs = (df['seq'] + PROMOTER_SEQ + RAND_BARCODE).tolist()
X = torch.stack([torch.from_numpy(sequence_to_onehot(c).astype(np.float32)).T for c in constructs])
print(f'{len(constructs)} seqs, X: {X.shape}')

5 seqs, X: torch.Size([5, 4, 281])


In [4]:
class MPRAHead(nn.Module):
    def __init__(self, n_positions=3, nl_size=1024, dropout=0.0,
                 activation='relu', pooling_type='flatten'):
        super().__init__()
        self.norm = nn.LayerNorm(ENCODER_DIM)
        in_dim = n_positions * ENCODER_DIM
        self.hidden_layers = nn.ModuleList([nn.Linear(in_dim, nl_size)])
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.output = nn.Linear(nl_size, 1)
        self.act = nn.ReLU()

    def forward(self, encoder_output):
        x = self.norm(encoder_output).flatten(1)
        for layer in self.hidden_layers:
            x = self.act(self.dropout(layer(x)))
        return self.output(x).squeeze(-1)


class AGModel(nn.Module):
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder, self.head = encoder, head

    def forward(self, x):
        x = x.transpose(1, 2)
        org = torch.zeros(x.shape[0], dtype=torch.long, device=x.device)
        enc = self.encoder(x, org, encoder_only=True)['encoder_output'].transpose(1, 2)
        return self.head(enc).unsqueeze(-1)


def load_model(model_name, ckpt_file='best_stage2.pt', device='cuda'):
    """Load encoder+head from a checkpoint."""
    ckpt_path = os.path.join(RESULTS_DIR, model_name, 'checkpoints', ckpt_file)
    enc = AlphaGenome.from_pretrained(WEIGHTS_PATH, device='cpu')
    remove_all_heads(enc)
    hd = MPRAHead()
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in ckpt:
        enc.load_state_dict(ckpt['model_state_dict'], strict=False)
    hd.load_state_dict(ckpt['head_state_dict'])
    return AGModel(enc, hd).to(device).eval()

In [5]:
def get_attr(model_name, ckpt_file='best_stage2.pt'):
    model = load_model(model_name, ckpt_file)
    attr = deep_lift_shap(
        model, X, target=0, n_shuffles=N_SHUFFLES, batch_size=20,
        device='cuda', additional_nonlinear_ops={AGCustomGELU: _nonlinear},
        warning_threshold=0.01, random_state=42, verbose=True,
    ).cpu().numpy()
    del model; torch.cuda.empty_cache()
    return attr

print(f'--- Best: {MODEL_NAME} ({BEST_CKPT}) ---')
attr_best = get_attr(MODEL_NAME, BEST_CKPT)

print(f'\n--- Overfit: {MODEL_NAME} ({OVERFIT_CKPT}) ---')
attr_overfit = get_attr(MODEL_NAME, OVERFIT_CKPT)

--- Best: K562_twostep_v4_do075 (best_stage2.pt) ---


  0%|          | 0/100 [00:00<?, ?it/s]/grid/wsbs/home_norepl/pmantill/LentiMPRA_mcs/alphagenome_torch_MPRAMoCon/interp/tangermeme/tangermeme/deep_lift_shap.py:465: RuntimeWarning: Convergence deltas too high: tensor([1.0714, 0.2487, 0.9632, 0.3393, 0.8099, 0.2966, 0.9234, 1.0542, 0.1498,
        0.9263, 0.8268, 1.2920, 1.1131, 0.8307, 0.8218, 1.0247, 0.8962, 0.5238,
        0.9010, 0.7500], device='cuda:0', grad_fn=<AbsBackward0>)
  warnings.warn("Convergence deltas too high: " +
 20%|██        | 20/100 [00:00<00:03, 22.39it/s]/grid/wsbs/home_norepl/pmantill/LentiMPRA_mcs/alphagenome_torch_MPRAMoCon/interp/tangermeme/tangermeme/deep_lift_shap.py:465: RuntimeWarning: Convergence deltas too high: tensor([0.8011, 0.7461, 0.4895, 0.7550, 0.6304, 1.2319, 0.6201, 1.0275, 0.7136,
        0.7361, 0.8738, 0.3650, 1.0610, 0.7784, 0.4871, 0.7867, 0.6536, 0.8193,
        1.1517, 0.4620], device='cuda:0', grad_fn=<AbsBackward0>)
  warnings.warn("Convergence deltas too high: " +
 40%|████      | 40


--- Overfit: K562_twostep_v4_do075 (latest_stage2.pt) ---


FileNotFoundError: [Errno 2] No such file or directory: '/grid/wsbs/home_norepl/pmantill/LentiMPRA_mcs/alphagenome_torch_MPRAMoCon/training/results/K562_twostep_v4_do075/checkpoints/latest_stage2.pt'

In [ ]:
# cleaned = best - (overfit - best) = 2*best - overfit
attr_cleaned = 2 * attr_best - attr_overfit

labels = ['Best model', 'Overfit model', 'Cleaned (best - diff)']
attrs = [attr_best, attr_overfit, attr_cleaned]

for si in range(len(constructs)):
    fig, axes = plt.subplots(3, 1, figsize=(20, 6))
    # Use consistent y-axis across rows
    vmax = max(np.abs(a[si, :, :230]).max() for a in attrs[:2]) * 1.1

    for row, (label, attr) in enumerate(zip(labels, attrs)):
        plot_logo(attr[si:si+1, :, :230], ax=axes[row])
        axes[row].set_ylim(-vmax, vmax)
        axes[row].set_ylabel(label, fontsize=9)
        if row < 2:
            axes[row].set_xticks([])

    axes[-1].set_xlabel('Enhancer position (bp)')
    fig.suptitle(f'Seq {si} — {CELL_TYPE}: best (ep6) vs overfit (ep11)', fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
# Per-position importance correlation: best vs overfit
from scipy.stats import pearsonr

for si in range(len(constructs)):
    imp_best = attr_best[si, :, :230].sum(axis=0)
    imp_over = attr_overfit[si, :, :230].sum(axis=0)
    imp_clean = attr_cleaned[si, :, :230].sum(axis=0)
    r_bo, _ = pearsonr(imp_best, imp_over)
    r_bc, _ = pearsonr(imp_best, imp_clean)
    print(f'Seq {si}: best-overfit r={r_bo:.3f}, best-cleaned r={r_bc:.3f}')